## Hugging Face CLI

In [2]:
import getpass
print("Enter you Hugging Face token:")
TOKEN = getpass.getpass()

Enter you Hugging Face token:
··········


In [3]:
!git config --global credential.helper store
!huggingface-cli login --token $TOKEN --add-to-git-credential

Token is valid (permission: read).
Your token has been saved in your configured git credential helpers (store).
Your token has been saved to /root/.cache/huggingface/token
Login successful




## Import the modules


In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

In [5]:
# Suppressing “INFO” and “WARNING” messages by setting the verbosity of the Transformers library.
from transformers import logging
logging.set_verbosity_error()

# Suppressing Python warnings
import warnings
warnings.filterwarnings("ignore")

## Unquantized Llama 3.1

### Load the model

In [6]:
model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"
model = AutoModelForCausalLM.from_pretrained(model_name,
                    device_map = "auto")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

### Data type of model's parameters

In [7]:
param_dtypes = [param.dtype for param in model.parameters()]
print("Parameter dtypes:", param_dtypes)

Parameter dtypes: [torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.

### Memory footprints

In [8]:
print(model.get_memory_footprint())

32121053440


### Inference of unquantized model

The session time for Jupyter notebook is 15 minutes on our platform. Executing the below cell takes around 45-50 minutes to show the output.
You can uncomment the code and execute on GPU enable machine to see the response.

In [ ]:
#tokenizer = AutoTokenizer.from_pretrained(model_name)
#input = tokenizer("Portugal is", return_tensors="pt").to('cuda')

#response = model.generate(**input, max_new_tokens = 50)
#print(tokenizer.batch_decode(response, skip_special_tokens=True))

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


['Portugal is a country located in southwestern Europe, bordered by Spain to the east and north, and the Atlantic Ocean to the west and south. It has a long history of colonialism and trade, and its culture reflects its rich heritage. Here are some key facts']


## Cleaning the memory

We are cleaning some variables and memory to avoid memory problems while executing the code.

In [9]:
import gc
del model
gc.collect()
torch.cuda.empty_cache()

## Implementing 8 bit quantization



### BitsAndBytes configuration





In [10]:
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True
)

### Load the model

In [11]:
model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"
quantized_model = AutoModelForCausalLM.from_pretrained(model_name,
                    quantization_config=bnb_config,
                    device_map = "auto")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

### Data type of model's parameters

In [12]:
param_dtypes = [param.dtype for param in quantized_model.parameters()]
print("Parameter dtypes:", param_dtypes)

Parameter dtypes: [torch.float16, torch.int8, torch.int8, torch.int8, torch.int8, torch.int8, torch.int8, torch.int8, torch.float16, torch.float16, torch.int8, torch.int8, torch.int8, torch.int8, torch.int8, torch.int8, torch.int8, torch.float16, torch.float16, torch.int8, torch.int8, torch.int8, torch.int8, torch.int8, torch.int8, torch.int8, torch.float16, torch.float16, torch.int8, torch.int8, torch.int8, torch.int8, torch.int8, torch.int8, torch.int8, torch.float16, torch.float16, torch.int8, torch.int8, torch.int8, torch.int8, torch.int8, torch.int8, torch.int8, torch.float16, torch.float16, torch.int8, torch.int8, torch.int8, torch.int8, torch.int8, torch.int8, torch.int8, torch.float16, torch.float16, torch.int8, torch.int8, torch.int8, torch.int8, torch.int8, torch.int8, torch.int8, torch.float16, torch.float16, torch.int8, torch.int8, torch.int8, torch.int8, torch.int8, torch.int8, torch.int8, torch.float16, torch.float16, torch.int8, torch.int8, torch.int8, torch.int8, torch.

### Memory footprints

In [13]:
print(quantized_model.get_memory_footprint())

9081209088


### Inference of 8-bit quantized model

In [14]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
input = tokenizer("Portugal is", return_tensors="pt").to('cuda')

response = quantized_model.generate(**input, max_new_tokens = 50)
print(tokenizer.batch_decode(response, skip_special_tokens=True))

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

['Portugal is a country located in the westernmost part of Europe, bordered by the Atlantic Ocean to the west and the Iberian Peninsula to the east. It has a rich history and culture, and is known for its beautiful beaches, delicious cuisine, and vibrant']
